In [1]:
#!/usr/bin/env python
# coding: utf-8

import dfBasics
import encoder
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, StringType
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
from datetime import time
import datetime as dt
import calendar
import pytz
from pyspark.sql.types import IntegerType
import numpy as np
import encoder
from pyspark.sql import functions as F
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import dfBasics
import pandas as pd
from os import listdir
import pyspark.sql.functions as f
import os.path

# Column setup
columns = ['CGLOBALMESSAGEID', 'CSTARTTIME', 'CENDTIME', 'CSTATUS', 'CSERVICE', 'CSENDERENDPOINTID', 
           'CSENDERPROTOCOL', 'CINBOUNDSIZE', 'CRECEIVERPROTOCOL', 'CRECEIVERENDPOINTID', 
           'CSLATAT', 'CMESSAGETAT2', 'CSLADELIVERYTIME']

# Spark session initialization
sparkSession = dfBasics.getSparkSession()

# Load parquet data
df = sparkSession.read.parquet('hdfs://172.30.17.145:8020/sla_sql_data/*/*').select(columns)
sender_receivers_df = pd.read_parquet('/home/jovyan/work/output/v00004/single/sender_receivers.parquet')

# UDFs for date processing and transformation
de = pytz.timezone('Europe/Berlin')

def date(x):
    return dt.datetime.fromtimestamp(float(x) / 1e3, tz=de)

udf_add_year = F.udf(lambda z: date(z).date().year, IntegerType())
udf_add_month = F.udf(lambda z: date(z).date().month, IntegerType())
udf_add_day = F.udf(lambda z: date(z).date().day, IntegerType())
udf_add_hour = F.udf(lambda z: date(z).time().hour, IntegerType())
udf_add_minute = F.udf(lambda z: date(z).time().minute, IntegerType())

# Encoder setup
def load_encoders(columns, npy_path):
    encoders = {}
    for column in columns:
        _encoder = encoder.TolerantLabelEncoder(ignore_unknown=True)
        _encoder.classes_ = np.load(f'{npy_path}/{column}.npy', allow_pickle=True)
        encoders[column] = _encoder
    return encoders

def e_transform(value, _encoder):
    try:
        return int(_encoder.transform([value])[0])
    except:
        return -1

# Function for encoding columns in Spark DataFrame
def encode_columns_spark(dataframe, columns, encoders):
    for column in columns:
        _encoder = encoders[column]
        udf_transform = F.udf(lambda z: e_transform(z, _encoder), StringType())
        dataframe = dataframe.withColumn(column, udf_transform(F.col(column)).cast("Integer"))
    return dataframe

# Function to process the data
def process(sender, receiver, dataframe, year, encoders):
    df3 = dataframe.withColumn("timestamp", F.from_unixtime(F.col("CSTARTTIME") / 1000))
    df3 = df3.withColumn("tyear", F.year("timestamp"))
    
    # Filter based on conditions
    df3 = df3.filter(
        (F.col("tyear") == year) & 
        (F.col("CSENDERENDPOINTID") == sender) & 
        (F.col("CRECEIVERENDPOINTID") == receiver)
    ).fillna(-1)

    # Encode columns and add date components
    df3 = encode_columns_spark(df3, columns, encoders)
    df3 = df3.withColumn("year", udf_add_year(F.col("CSTARTTIME")))\
             .withColumn("month", udf_add_month(F.col("CSTARTTIME")))\
             .withColumn("day", udf_add_day(F.col("CSTARTTIME")))\
             .withColumn("hour", udf_add_hour(F.col("CSTARTTIME")))\
             .withColumn("minute", udf_add_minute(F.col("CSTARTTIME")))
    
    # Cast required columns to 'long' type
    long_columns = ['CSTARTTIME', 'CENDTIME', 'CINBOUNDSIZE', 'CSLATAT', 'CMESSAGETAT2', 'CSLADELIVERYTIME']
    df3 = df3.select(*df3.columns, *[F.col(col).cast(LongType()).alias(col) for col in long_columns])
    
    return df3

# Main processing loop
ENCODED_PATH = '/home/jovyan/work/output/v00004/v00000/encoded/parts/'
NPY_PATH = '/home/jovyan/work/output/v00004/npy/'
years = [2019, 2020, 2021, 2022, 2023, 2024]

encoders = load_encoders(['CSTATUS', 'CSERVICE', 'CSENDERENDPOINTID', 'CSENDERPROTOCOL', 
                          'CRECEIVERPROTOCOL', 'CRECEIVERENDPOINTID'], NPY_PATH)

columns = ['CSTATUS','CSERVICE','CSENDERENDPOINTID','CSENDERPROTOCOL','CRECEIVERPROTOCOL','CRECEIVERENDPOINTID']

for index, row in sender_receivers_df.iterrows():
    enc_sender = index
    enc_receivers = list(row['CRECEIVERENDPOINTID'])
    sender = e_transform(enc_sender, encoders['CSENDERENDPOINTID'])
    
    for enc_receiver in enc_receivers:
        receiver = e_transform(enc_receiver, encoders['CRECEIVERENDPOINTID'])
        
        for year in years:
            filename = f"{ENCODED_PATH}sla_enc_srfull_v00004_v00000_{sender}_{receiver}_{year}_0.parquet"
            if not os.path.isfile(f'{filename}/_SUCCESS'):
                df_processed = process(sender=sender, receiver=receiver, dataframe=df, year=year, encoders=encoders)
                df_processed.write.mode("overwrite").parquet(filename)


AnalysisException: Found duplicate column(s) when inserting into file:/home/jovyan/work/output/v00004/v00000/encoded/parts/sla_enc_srfull_v00004_v00000_-1_-1_2019_0.parquet: `cendtime`, `cinboundsize`, `cmessagetat2`, `csladeliverytime`, `cslatat`, `cstarttime`

In [ ]:
encoders